# 🚀 Day 26 — XGBoost
## Credit Card Default Risk Prediction

**Objective:** Predict credit-card default risk using XGBoost and compare it with Logistic Regression and Random Forest.

**Dataset:** UCI Default of Credit Card Clients


## 📌 Google Colab Setup
Upload the official `default of credit card clients.xls` file in the next cell.

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))


In [ ]:
!pip -q install xgboost openpyxl

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, roc_curve
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
print("Libraries imported successfully!")


## 1. Load Dataset

In [ ]:
DATA_PATH = Path("default of credit card clients.xls")
if not DATA_PATH.exists():
    raise FileNotFoundError("Upload default of credit card clients.xls first.")

df = pd.read_excel(DATA_PATH, header=1)
print("Dataset shape:", df.shape)
display(df.head())


## 2. Rename Columns

In [ ]:
df.columns = [
    "ID","LIMIT_BAL","SEX","EDUCATION","MARRIAGE","AGE",
    "PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6",
    "BILL_AMT1","BILL_AMT2","BILL_AMT3","BILL_AMT4","BILL_AMT5","BILL_AMT6",
    "PAY_AMT1","PAY_AMT2","PAY_AMT3","PAY_AMT4","PAY_AMT5","PAY_AMT6",
    "DEFAULT"
]
print(df.columns.tolist())


## 3. Data Understanding

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing Values"))
print("\nDuplicate rows:", df.duplicated().sum())


## 4. Data Cleaning

In [ ]:
df = df.drop_duplicates().copy()
df = df.drop(columns=["ID"])
print("Shape after cleaning:", df.shape)


## 5. Target Analysis

In [ ]:
target_counts = df["DEFAULT"].value_counts().sort_index()
display(target_counts.to_frame("Customers"))
print("Target percentages:")
display((df["DEFAULT"].value_counts(normalize=True) * 100).round(2))

plt.figure(figsize=(6,4))
plt.bar(["No Default","Default"], [target_counts.get(0,0), target_counts.get(1,0)])
plt.title("Credit Card Default Distribution")
plt.ylabel("Customers")
plt.show()


## 6. Feature Engineering

In [ ]:
bill_cols = ["BILL_AMT1","BILL_AMT2","BILL_AMT3","BILL_AMT4","BILL_AMT5","BILL_AMT6"]
pay_cols = ["PAY_AMT1","PAY_AMT2","PAY_AMT3","PAY_AMT4","PAY_AMT5","PAY_AMT6"]
delay_cols = ["PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]

df["AVG_BILL_AMT"] = df[bill_cols].mean(axis=1)
df["AVG_PAY_AMT"] = df[pay_cols].mean(axis=1)
df["TOTAL_PAY_AMT"] = df[pay_cols].sum(axis=1)
df["TOTAL_BILL_AMT"] = df[bill_cols].sum(axis=1)
df["PAYMENT_BILL_RATIO"] = df["AVG_PAY_AMT"] / (df["AVG_BILL_AMT"].abs() + 1)
df["MAX_PAYMENT_DELAY"] = df[delay_cols].max(axis=1)
df["DELAYED_MONTHS"] = (df[delay_cols] > 0).sum(axis=1)

print("Feature engineering completed.")
display(df.head())


## 7. Prepare Data

In [ ]:
X = df.drop(columns=["DEFAULT"])
y = df["DEFAULT"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 8. Handle Class Imbalance

In [ ]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
scale_pos_weight = negative_count / positive_count

print("Negative class:", negative_count)
print("Positive class:", positive_count)
print(f"scale_pos_weight: {scale_pos_weight:.2f}")


## 9. Logistic Regression Baseline

In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ))
])

logistic_model.fit(X_train, y_train)
logistic_pred = logistic_model.predict(X_test)
logistic_prob = logistic_model.predict_proba(X_test)[:, 1]
print("Logistic Regression trained.")


## 10. Random Forest

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=200, max_depth=12, min_samples_split=5,
    min_samples_leaf=2, class_weight="balanced",
    random_state=RANDOM_STATE, n_jobs=-1
)
random_forest.fit(X_train, y_train)
rf_pred = random_forest.predict(X_test)
rf_prob = random_forest.predict_proba(X_test)[:, 1]
print("Random Forest trained.")


## 11. XGBoost ⭐

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=2,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]
print("XGBoost trained successfully!")


## 12. Model Evaluation

In [ ]:
def evaluate_model(name, y_true, pred, prob):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1 Score": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, prob)
    }

results_df = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, logistic_pred, logistic_prob),
    evaluate_model("Random Forest", y_test, rf_pred, rf_prob),
    evaluate_model("XGBoost", y_test, xgb_pred, xgb_prob)
]).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)

display(results_df.round(4))


## 13. Model Comparison

In [ ]:
results_df.set_index("Model")[["Accuracy","Precision","Recall","F1 Score","ROC-AUC"]].plot(
    kind="bar", figsize=(12,6)
)
plt.title("Credit Default Model Comparison")
plt.ylabel("Score")
plt.ylim(0,1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 14. XGBoost Classification Report

In [ ]:
print(classification_report(
    y_test, xgb_pred,
    target_names=["No Default","Default"],
    zero_division=0
))


## 15. XGBoost Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, xgb_pred)
print(cm)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Default","Default"]
).plot()
plt.title("XGBoost Confusion Matrix")
plt.show()


## 16. ROC Curve Comparison

In [ ]:
probabilities = {
    "Logistic Regression": logistic_prob,
    "Random Forest": rf_prob,
    "XGBoost": xgb_prob
}

plt.figure(figsize=(9,7))
for name, prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0,1], [0,1], linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Credit Default Prediction")
plt.legend()
plt.grid(True)
plt.show()


## 17. XGBoost Feature Importance

In [ ]:
feature_importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb_model.feature_importances_
}).sort_values("Importance", ascending=False).reset_index(drop=True)

display(feature_importance_df.head(15).round(6))


In [ ]:
top_features = feature_importance_df.head(15).sort_values("Importance")

plt.figure(figsize=(10,7))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top XGBoost Features")
plt.tight_layout()
plt.show()


## 18. Business Insights

- XGBoost can capture non-linear relationships between repayment behavior, bills, payments, and customer characteristics.
- Feature importance provides a starting point for understanding which variables contribute most to predictions.
- Recall and ROC-AUC should be considered alongside accuracy because default prediction involves class imbalance.
- In a real financial setting, predictions should support risk analysis and human review rather than automatically determining an individual's credit eligibility.

## 19. Save Results

In [ ]:
results_df.to_csv("Day26_XGBoost_Model_Results.csv", index=False)
feature_importance_df.to_csv("Day26_XGBoost_Feature_Importance.csv", index=False)

print("Saved:")
print("✓ Day26_XGBoost_Model_Results.csv")
print("✓ Day26_XGBoost_Feature_Importance.csv")


# 🎯 Key Learnings

1. XGBoost is a powerful gradient-boosting algorithm.
2. Learning rate and number of estimators control boosting behavior.
3. `max_depth`, `min_child_weight`, `subsample`, and `colsample_bytree` affect complexity and generalization.
4. `scale_pos_weight` helps address class imbalance.
5. ROC-AUC, precision, recall, and F1 provide a more complete evaluation than accuracy alone.
6. Feature importance helps connect model predictions with business interpretation.

**Day 26/30 — XGBoost completed! ⚡**